# 01 · FPSD-CNN AD–CN en Kaggle (GPU)
Verifica el bundle, ejecuta tests, mide 10 épocas y corre `forensic_test_selected` y `valid_subject_disjoint`. Un benchmark o una semilla parcial no se presentan como resultado completo.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, kagglehub
assert Path('/kaggle/working').exists() and os.environ.get('KAGGLE_KERNEL_RUN_TYPE'), 'Este notebook debe ejecutarse en Kaggle.'
REPO = Path('/kaggle/working/fpsd-cnn-reassessment')
if not (REPO/'pyproject.toml').exists():
    repo_candidates = [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml') if (p.parent/'src/fpsd_reassessment').exists()]
    preferred_repo = [p for p in repo_candidates if 'fpsd-cnn-reassessment-code' in str(p)]
    if len(preferred_repo) == 1:
        source_repo = preferred_repo[0]
    elif len(repo_candidates) == 1:
        source_repo = repo_candidates[0]
    else:
        source_repo = Path(kagglehub.dataset_download('sarahvasquez97/fpsd-cnn-reassessment-code'))
    assert (source_repo/'pyproject.toml').exists(), f'Código no encontrado en {source_repo}'
    shutil.copytree(source_repo, REPO, dirs_exist_ok=True)
feature_candidates = [p.parent for p in Path('/kaggle/input').glob('**/paper95_features.npz') if (p.parent/'SHA256SUMS.txt').exists()]
preferred_features = [p for p in feature_candidates if p.name == 'fpsd_features']
if len(preferred_features) == 1:
    FEATURES = preferred_features[0]
elif len(feature_candidates) == 1:
    FEATURES = feature_candidates[0]
else:
    feature_data = Path(kagglehub.dataset_download('sarahvasquez97/fpsd-paper95-features'))
    feature_candidates = [p.parent for p in feature_data.glob('**/paper95_features.npz') if (p.parent/'SHA256SUMS.txt').exists()]
    preferred_features = [p for p in feature_candidates if p.name == 'fpsd_features']
    assert len(preferred_features) == 1, f'Se esperaba un directorio fpsd_features, encontrados: {feature_candidates}'
    FEATURES = preferred_features[0]
print('Repositorio:', REPO, '\nCaracterísticas:', FEATURES)

In [ ]:
import importlib, json
sys.path.insert(0, str(REPO/'src'))
os.environ['PYTHONPATH'] = str(REPO/'src')
versions = {}
for name in ('numpy', 'scipy', 'pandas', 'sklearn', 'mne', 'pyarrow', 'yaml', 'psutil', 'matplotlib', 'seaborn', 'tqdm', 'pytest', 'torch'):
    module = importlib.import_module(name)
    versions[name] = getattr(module, '__version__', 'unknown')
print(json.dumps(versions, indent=2))

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'cuDNN:', torch.backends.cudnn.version())
assert torch.cuda.is_available(), 'Active una GPU en Notebook options → Accelerator.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)
from fpsd_reassessment.models import validate_architecture
assert validate_architecture() == 141732

In [ ]:
# Si el benchmark no cabe con margen, deje una sola semilla por versión: [2026], [2027] o [2028].
SEEDS = []  # vacío = decisión automática: tres semillas si caben; si no, sólo 2026
cmd = [sys.executable, str(REPO/'scripts/train_baseline.py'), '--features', str(FEATURES), '--config', str(REPO/'configs/experiments/baseline_ad_cn.yaml'), '--output', '/kaggle/working/week1_results']
for seed in SEEDS: cmd += ['--seed', str(seed)]
subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
RESULTS = Path('/kaggle/working/week1_results')
subprocess.run([sys.executable, str(REPO/'scripts/evaluate.py'), str(RESULTS)], cwd=REPO, check=True)
# El reporte sólo se materializa cuando hay tablas; ejecuciones parciales permanecen explícitas.
subprocess.run([sys.executable, str(REPO/'scripts/generate_report.py'), str(RESULTS), '--report', str(REPO/'reports/week1_baseline.md')], cwd=REPO, check=True)
subprocess.run([sys.executable, str(REPO/'scripts/export_results.py'), str(RESULTS), '--repo', str(REPO)], cwd=REPO, check=True)
print('Completo:', '/kaggle/working/week1_results_bundle.zip')
print('Para GitHub:', '/kaggle/working/github_results_bundle.zip')